# Vacher comparison rerun with the corrected loader, 50 pairs (`tab:app-vacher` images)

Produces `corrected_rerun/{lit_manifest.csv, lit_midpoints/, lit_analysis_n25.csv, lit_analysis_n50.csv}` -- the images and the Wilcoxon p>.25 result behind `tab:app-vacher`.

**Needs:** DTD, the CNT checkpoint, the Vacher baseline repo (`install/vacher.sh`).

**Maintained runnable path:** `experiments/vacher_50.py` reproduces the FID column of this comparison. This notebook remains the only record of the Wilcoxon significance test and the exact 50-pair selection.

In [ ]:
!pip install -q pot
import os, sys, torch, torch.nn.functional as F, numpy as np, glob, random, inspect, time
from pathlib import Path
from PIL import Image
import pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from google.colab import drive
drive.mount('/content/drive')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
DTD_ROOT = f'{BASE}/dtd/images'
OUT = f'{BASE}/corrected_rerun'; os.makedirs(OUT, exist_ok=True)
TMP = '/content/tmp_imgs'; os.makedirs(TMP, exist_ok=True)

os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT_DIR='/content/cnt-siga24'; os.chdir(CNT_DIR); sys.path.insert(0,CNT_DIR)
from src.inference.core import TexAutoEncoderInference, load_image
model = TexAutoEncoderInference(ckpt_path=Path(f"{CNT_DIR}/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
FMS = [round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]

# what does the transform actually do?
from src.inference import core as cnt_core
try:
    print(inspect.getsource(cnt_core.build_content_transform))
except Exception as e:
    import src.inference.core as c
    print("could not fetch:", e)

# ---- THE loader. Always go through load_image, via a temp file if needed. ----
_tmp_counter = [0]
def cnt_load(x, size=IMG_SIZE):
    """x: path or PIL. Always routed through CNT's load_image."""
    if isinstance(x, (str, Path)):
        return load_image(Path(x), size).to(device)
    _tmp_counter[0] += 1
    fp = f'{TMP}/t{_tmp_counter[0] % 64}.png'
    x.convert('RGB').save(fp)
    return load_image(Path(fp), size).to(device)

def to_pil(dec):
    a = ((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))
def contrast(im): return float(np.asarray(im.convert('L')).std())

def get_gen_inputs(img):
    with torch.no_grad():
        blob = model.encode(img); layout = model.splat(blob)
        return blob, layout["layer_level_layouts"][-1][0]["blob_features"].clone(), \
                     layout["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(blob_ref, bf, si):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        layout["layer_level_layouts"][-1][0]["blob_features"] = bf
        layout["layer_level_layouts"][-1][0]["score_img"] = si
        return model.decode(layout)
def build_style_maps(bf, si, fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref, sm):
    with torch.no_grad():
        layout=model.splat(blob_ref); si=layout["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        feat_in=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=feat_in.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def sinkhorn_transport(src,tgt,eps=0.05,n_iters=100):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=src.device)/N; b=torch.ones(M,1,device=src.device)/M; u=torch.ones(N,1,device=src.device)
    for _ in range(n_iters): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    plan=u*K*v.T; plan=plan/(plan.sum(1,keepdim=True)+1e-10); return plan@tgt

def build_d(ia, ib):
    ba,bfa,sia = get_gen_inputs(ia); bb,bfb,sib = get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def interp_ot(d, eta, n_sub=4096):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); src,tgt=sm_a[L][0],sm_b[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    i_s=torch.randperm(N,device=device)[:n_sub]; i_t=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    interp=((1-eta)*sf+eta*tt[nn]).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    sm={s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
    return decode_with_style_maps(d["blob_a"], sm)
def interp_ot_sym(d, eta, n_sub=4096):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sm_a[L][0],sm_b[L][0]; C,H,W=A.shape
    af=A.permute(1,2,0).reshape(-1,C); bfm=B.permute(1,2,0).reshape(-1,C); N=af.shape[0]
    ia=torch.randperm(N,device=device)[:n_sub]; ib=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad():
        t_ab=sinkhorn_transport(af[ia],bfm[ib]); t_ba=sinkhorn_transport(bfm[ib],af[ia])
    nn_a=torch.cdist(af,af[ia]).argmin(1); nn_b=torch.cdist(bfm,bfm[ib]).argmin(1)
    interp=(0.5*(((1-eta)*af+eta*t_ab[nn_a])+(eta*bfm+(1-eta)*t_ba[nn_b]))).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    sm={s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
    return decode_with_style_maps(d["blob_a"], sm)
def interp_lin_pixel(d, eta):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"], {s:(1-eta)*sm_a[s]+eta*sm_b[s] for s in set(FMS)})
def interp_lin_texton(d, eta):
    return decode_interp(d["blob_a"], (1-eta)*d["bf_a"]+eta*d["bf_b"], (1-eta)*d["si_a"]+eta*d["si_b"])

FNS = {'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton,
       'ot':interp_ot,'ot_sym':interp_ot_sym}
METHODS = list(FNS)
random.seed(1)
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
print("\nsetup done on", device)
os.chdir('/content')
os.system('rm -rf texture-interpolation')
os.system('git clone -q --depth 1 https://github.com/JonathanVacher/texture-interpolation.git')
VACHER_DIR = '/content/texture-interpolation/texture-synthesis-algorithm'
os.chdir(CNT_DIR)
print("Vacher models:", os.listdir(f'{VACHER_DIR}/models'))

from torchvision.models import inception_v3, Inception_V3_Weights
import timm
_inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True).to(device).eval()
_inc.fc = torch.nn.Identity()
dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).to(device).eval()
for p in list(_inc.parameters())+list(dino.parameters()): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
_S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
@torch.no_grad()
def _feats(net,size,pils,bs=24):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255.
            for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device)
        b=F.interpolate(b,size=size,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(net(b).cpu().numpy())
    return np.concatenate(out)
inc_feats  = lambda p: _feats(_inc,299,p)
dino_feats = lambda p: _feats(dino,518,p)
print("observers ready")

In [ ]:
def vacher_interp(pathA, pathB, weight=0.5, model_name='tex_wasser',
                  n_iter=30, bfgs_iter=20, shape='256,256', seed=0):
    """Vacher's optimisation-based interpolation. Inputs must be equal-sized."""
    cwd = os.getcwd()
    os.chdir(VACHER_DIR)
    if VACHER_DIR not in sys.path: sys.path.insert(0, VACHER_DIR)
    for mod in [m for m in list(sys.modules) if m.startswith(('models','options'))]:
        del sys.modules[mod]
    from options.base_options import BaseOptions
    from models import create_model, utils
    argv_bak = sys.argv
    sys.argv = ['run_synthesis.py','--input_tex_1',str(pathA),'--input_tex_2',str(pathB),
                '--weight_1',str(weight),'--model',model_name,
                '--n_iter',str(n_iter),'--bfgs_iter',str(bfgs_iter),
                '--shape',shape,'--seed',str(seed),'--display_freq','100000']
    try:
        opt = BaseOptions().parse()
        m = create_model(opt); m.get_net_and_loss()
        for _ in range(opt.n_iter+1): m.optimize_parameters()
        arr = utils.tensor2im(m.output_tex.data.clamp_(0,1))
        return Image.fromarray(arr.astype(np.uint8)).convert('RGB').resize((IMG_SIZE,)*2)
    finally:
        sys.argv = argv_bak; os.chdir(cwd)

TMPV = '/content/vacher_tmp'; os.makedirs(TMPV, exist_ok=True)
def prep_square(path, out):
    Image.open(path).convert('RGB').resize((IMG_SIZE,)*2, Image.BILINEAR).save(out)
    return out

pa, pb = dtd_path(cats[3],11), dtd_path(cats[17],22)
qa, qb = prep_square(pa,f'{TMPV}/A.png'), prep_square(pb,f'{TMPV}/B.png')
for mn in ['tex_wasser','tex_gram','tex_gauss']:
    t0=time.time()
    try:
        img = vacher_interp(qa,qb,0.5,mn,n_iter=30)
        print(f"  {mn:12} OK  {time.time()-t0:5.1f}s  contrast {contrast(img):.1f}")
    except Exception as e:
        print(f"  {mn:12} FAILED: {type(e).__name__}: {str(e)[:120]}")

In [ ]:
N_TOTAL = 50
BATCH   = (25, 50)          # second run: (25, 50)
N_ITER  = 30
IMGDIR  = f'{OUT}/lit_midpoints'; os.makedirs(IMGDIR, exist_ok=True)
VACHER  = ['tex_wasser','tex_gram','tex_gauss']
ALL_M   = METHODS + VACHER

rngp = random.Random(4242)
PAIRS_LIT = []
while len(PAIRS_LIT) < N_TOTAL:
    a,b = rngp.sample(cats,2)
    PAIRS_LIT.append((dtd_path(a,20000+len(PAIRS_LIT)), dtd_path(b,40000+len(PAIRS_LIT))))

manifest = f'{OUT}/lit_manifest.csv'
done = set()
if os.path.exists(manifest):
    md = pd.read_csv(manifest); done = {(int(r.pair), r.method) for r in md.itertuples()}
    print(f"resuming: {len(done)} already generated")

recs=[]; t0=time.time()
for k in range(*BATCH):
    pa,pb = PAIRS_LIT[k]
    qa,qb = prep_square(pa,f'{TMPV}/A.png'), prep_square(pb,f'{TMPV}/B.png')
    Image.open(qa).save(f'{IMGDIR}/pair{k:03d}_A.png')
    Image.open(qb).save(f'{IMGDIR}/pair{k:03d}_B.png')
    d = None
    for m in ALL_M:
        if (k,m) in done: continue
        fp = f'{IMGDIR}/pair{k:03d}_{m}.png'; ts=time.time()
        try:
            if m in FNS:
                if d is None: d = build_d(cnt_load(qa), cnt_load(qb))
                img = to_pil(FNS[m](d, 0.5))
            else:
                img = vacher_interp(qa, qb, 0.5, m, n_iter=N_ITER)
            img.save(fp)
            recs.append(dict(pair=k, method=m, path=fp, secs=time.time()-ts, ok=True))
        except Exception as e:
            print(f"    pair {k} {m} FAILED: {type(e).__name__}: {str(e)[:100]}")
            recs.append(dict(pair=k, method=m, path='', secs=time.time()-ts, ok=False))
    new = pd.DataFrame(recs)
    if os.path.exists(manifest): new = pd.concat([pd.read_csv(manifest), new], ignore_index=True)
    new.to_csv(manifest, index=False); recs=[]
    el=time.time()-t0; per=el/(k-BATCH[0]+1)
    print(f"  pair {k:3d}  ({el/60:5.1f}m, {per:.0f}s/pair, ~{per*(BATCH[1]-k-1)/60:.0f}m left)")

md = pd.read_csv(manifest)
print(f"\n{len(md)} images over {md.pair.nunique()} pairs")
print(md.groupby('method').agg(n=('ok','size'), ok=('ok','sum'), mean_s=('secs','mean')).round(1))

In [ ]:
md = pd.read_csv(manifest); md = md[md.ok]

# manifold from raw DTD textures
mimgs = [Image.open(dtd_path(random.choice(cats),50000+i)).convert('RGB').resize((IMG_SIZE,)*2)
         for i in range(400)]
def make_realism(featfn, imgs, K=3):
    Phi = featfn(imgs); Phi = Phi[~np.isnan(Phi).any(axis=1)]
    D = np.linalg.norm(Phi[:,None]-Phi[None,:],axis=2); np.fill_diagonal(D,np.inf)
    rad = np.sort(D,axis=1)[:,K-1]; keep = rad<=np.median(rad)
    Pr, rr = Phi[keep], rad[keep]
    def r(f):
        dd=np.linalg.norm(Pr-f,axis=1)+1e-9; return float(np.max(rr/dd))
    return r
realism_inc  = make_realism(inc_feats,  mimgs)
realism_dino = make_realism(dino_feats, mimgs)

rows=[]
for k in sorted(md.pair.unique()):
    have = {r.method: r.path for r in md[md.pair==k].itertuples()}
    if len(have) < len(ALL_M): continue
    A = Image.open(f'{IMGDIR}/pair{k:03d}_A.png'); B = Image.open(f'{IMGDIR}/pair{k:03d}_B.png')
    imgs = [Image.open(have[m]) for m in ALL_M]
    fI = inc_feats(imgs); fD = dino_feats(imgs)
    fA_d, fB_d = dino_feats([A,B])
    rec = dict(pair=k, c_src=0.5*(contrast(A)+contrast(B)))
    for m,fi,fd,im in zip(ALL_M, fI, fD, imgs):
        rec[f'Rinc_{m}']  = realism_inc(fi)
        rec[f'Rdino_{m}'] = realism_dino(fd)
        dA = float(np.linalg.norm(fd-fA_d)); dB = float(np.linalg.norm(fd-fB_d))
        rec[f'bal_{m}']      = abs(dA-dB)/(dA+dB+1e-9)
        rec[f'struct_{m}']   = 0.5*(dA+dB)/(np.linalg.norm(fA_d-fB_d)+1e-9)
        rec[f'contrast_{m}'] = contrast(im)
    rows.append(rec)

dfl = pd.DataFrame(rows); dfl.to_csv(f'{OUT}/lit_analysis_n{len(dfl)}.csv', index=False)

print(f"=== LITERATURE BASELINES, corrected (n={len(dfl)}), source contrast {dfl.c_src.median():.1f} ===")
print(f"  {'method':16} {'R inc↑':>8} {'R dino↑':>9} {'%R≥1 dino':>11} {'balance↓':>10} {'struct↓':>9} {'contrast':>9}")
for m in ALL_M:
    print(f"  {m:16} {dfl[f'Rinc_{m}'].median():8.3f} {dfl[f'Rdino_{m}'].median():9.3f} "
          f"{100*(dfl[f'Rdino_{m}']>=1).mean():10.0f}% {dfl[f'bal_{m}'].median():10.3f} "
          f"{dfl[f'struct_{m}'].median():9.3f} {dfl[f'contrast_{m}'].median():9.1f}")

print(f"\n=== ours vs Vacher (paired) ===")
for tgt in ['ot','ot_sym']:
    print(f"  --- {tgt} vs ---")
    for base in VACHER:
        line = f"    {base:12}"
        for key,alt,lab in [('Rdino_','greater','realism'), ('bal_','less','balance'),
                            ('struct_','less','struct')]:
            a,b = dfl[f'{key}{tgt}'], dfl[f'{key}{base}']
            w=int(((a>b) if alt=='greater' else (a<b)).sum())
            p=wilcoxon(a,b,alternative=alt).pvalue
            line += f"  {lab:8}{w:3d}/{len(dfl)} p={p:.1e}"
        print(line)